# 第10章：张量并行通信与 GEMM 的协同 — TP 是如何和矩阵乘法配合的

## 本章内容

深入分析 vLLM 中 **Tensor Parallelism (TP)** 与矩阵乘法的配合机制：

1. 为什么多卡推理需要通信？
2. ColumnParallelLinear: 权重按列切 → GEMM → all-gather
3. RowParallelLinear: 权重按行切 → GEMM → all-reduce
4. Llama 模型的 TP 数据流：Attention + MLP 的通信编排
5. MoE 的 TP vs EP 通信
6. 通信与计算的重叠优化
7. 用 PyTorch 模拟完整的 TP2 GEMM 流程

## 前置知识
- Notebook 06-09 的矩阵乘法理解
- 基本的分布式通信概念 (all-reduce, all-gather)

## 10.1 为什么多卡推理需要通信？

### 问题：模型太大，一张卡放不下

```
Llama-70B 模型参数:
  hidden_size = 8192
  num_heads = 64
  num_kv_heads = 8
  intermediate_size = 28672
  num_layers = 80

  每层参数量 (不含 KV cache):
    QKV proj: 8192 × (64×128 + 8×128 + 8×128) = 8192 × 10240 ≈ 160 MB (FP16)
    O proj:   64×128 × 8192 = 8192 × 8192 ≈ 128 MB
    Gate+Up:  8192 × 28672 × 2 ≈ 896 MB
    Down:     28672 × 8192 ≈ 448 MB
    ─────────────────────────────────
    每层合计: ≈ 1.6 GB

  80 层总计: ≈ 128 GB → 单张 80GB A100 放不下!

解决方案: Tensor Parallelism (TP)
  TP2: 2 张卡，每卡存一半权重
  TP4: 4 张卡，每卡存 1/4 权重
  TP8: 8 张卡，每卡存 1/8 权重

  但是: 切分权重后，每卡的 GEMM 结果只是"部分"结果
        → 需要通信才能得到完整输出
```

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# 模拟参数
hidden_size = 4096
intermediate_size = 11008
num_heads = 32
head_dim = 128
tp_size = 2  # TP2: 两张卡

print(f"模拟 TP{tp_size} 场景:")
print(f"  hidden_size = {hidden_size}")
print(f"  intermediate_size = {intermediate_size}")
print(f"  每卡 intermediate_size = {intermediate_size // tp_size}")
print(f"  num_heads = {num_heads}")
print(f"  每卡 num_heads = {num_heads // tp_size}")
print()
print("核心问题: 切分权重后, 每卡 GEMM 只能算出部分结果")
print("        需要 all-reduce 或 all-gather 通信才能拼出完整输出")

## 10.2 Column Parallel Linear: 按列切分权重

### 数学原理

$$Y = X \cdot A \quad \text{其中 } A = [A_1 | A_2 | \cdots | A_p]$$

将权重矩阵 $A$ 沿**列方向** (输出维度) 切成 $p$ 份:

$$Y = X \cdot [A_1 | A_2 | \cdots | A_p] = [X A_1 | X A_2 | \cdots | X A_p]$$

- GPU 0 计算 $Y_0 = X \cdot A_0$ → 得到输出的前半列
- GPU 1 计算 $Y_1 = X \cdot A_1$ → 得到输出的后半列
- **不需要通信**就能各自算出部分结果！
- 如果后续需要完整输出，做一次 **all-gather** 即可

### 具体数值例子

$$X = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}, \quad A = \begin{bmatrix} 1 & 2 & 3 & 4 \\ 5 & 6 & 7 & 8 \end{bmatrix}$$

$$Y = X A = \begin{bmatrix} 11 & 14 & 17 & 20 \\ 23 & 30 & 37 & 44 \end{bmatrix}$$

TP2 切分:
$$A_0 = \begin{bmatrix} 1 & 2 \\ 5 & 6 \end{bmatrix}, \quad A_1 = \begin{bmatrix} 3 & 4 \\ 7 & 8 \end{bmatrix}$$

$$Y_0 = X A_0 = \begin{bmatrix} 11 & 14 \\ 23 & 30 \end{bmatrix}, \quad Y_1 = X A_1 = \begin{bmatrix} 17 & 20 \\ 37 & 44 \end{bmatrix}$$

$$Y = [Y_0 | Y_1] = \text{all-gather}(Y_0, Y_1) \checkmark$$

> **生活类比**: 像两个打字员分别打一份文件的左半边和右半边，最后把两页拼在一起。
> 每人只需看到完整的原稿 (X)，但各自只负责一半输出。

In [ ]:
# ======== Column Parallel Linear 模拟 ========

# 完整权重: [hidden_size, output_size]
W_full = torch.randn(hidden_size, intermediate_size)
X = torch.randn(8, hidden_size)  # 8 tokens

# 完整计算 (参考)
Y_ref = X @ W_full
print(f"完整 GEMM: X[{X.shape}] @ W[{W_full.shape}] → Y[{Y_ref.shape}]")

# ========== TP2 切分 ==========
# 每卡存一半列
shard_size = intermediate_size // tp_size
W_gpu0 = W_full[:, 0:shard_size]           # 前半列
W_gpu1 = W_full[:, shard_size:2*shard_size] # 后半列

# 每卡独立 GEMM (输入 X 不切, 每卡都有完整 X)
Y_gpu0 = X @ W_gpu0  # [8, 5504]
Y_gpu1 = X @ W_gpu1  # [8, 5504]

print(f"\nTP2 Column Parallel:")
print(f"  GPU 0: X[{X.shape}] @ W_0[{W_gpu0.shape}] → Y_0[{Y_gpu0.shape}]")
print(f"  GPU 1: X[{X.shape}] @ W_1[{W_gpu1.shape}] → Y_1[{Y_gpu1.shape}]")

# all-gather: 拼接
Y_gathered = torch.cat([Y_gpu0, Y_gpu1], dim=-1)  # [8, 11008]
print(f"  all-gather → Y[{Y_gathered.shape}]")

# 验证正确性
diff = (Y_gathered - Y_ref).abs().max()
print(f"  验证: max diff = {diff:.2e} ✓")

print(f"""
通信开销:
  all-gather 传输量 = {Y_gpu0.numel() * 2 / 1024:.1f} KB (每卡发送自己的部分)
  相比 GEMM 的计算量, 通信通常很小
  但在 NVLink 上仍有 ~microseconds 延迟
""")

## 10.3 Row Parallel Linear: 按行切分权重

### 数学原理

$$Y = X \cdot A + b \quad \text{其中 } A = \begin{bmatrix} A_1 \\ A_2 \\ \vdots \\ A_p \end{bmatrix}, \quad X = [X_1 | X_2 | \cdots | X_p]$$

将权重矩阵沿**行方向** (输入维度) 切分，**同时输入也沿最后一维切分**:

$$Y = \sum_{i=1}^{p} X_i \cdot A_i$$

- GPU 0 计算 $Y_0 = X_0 \cdot A_0$ → 得到**部分和**
- GPU 1 计算 $Y_1 = X_1 \cdot A_1$ → 得到**部分和**
- **all-reduce (sum)** 才能得到完整结果: $Y = Y_0 + Y_1$

### 具体数值例子

$$X = [1, 2, 3, 4], \quad A = \begin{bmatrix} 1 & 2 \\ 3 & 4 \\ 5 & 6 \\ 7 & 8 \end{bmatrix}$$

$$Y = X A = [1{\cdot}1+2{\cdot}3+3{\cdot}5+4{\cdot}7,\; 1{\cdot}2+2{\cdot}4+3{\cdot}6+4{\cdot}8] = [50, 60]$$

TP2 切分:
$$X_0 = [1, 2], X_1 = [3, 4], \quad A_0 = \begin{bmatrix}1 & 2\\3 & 4\end{bmatrix}, A_1 = \begin{bmatrix}5 & 6\\7 & 8\end{bmatrix}$$

$$Y_0 = X_0 A_0 = [7, 10], \quad Y_1 = X_1 A_1 = [43, 50]$$

$$Y = Y_0 + Y_1 = [50, 60] = \text{all-reduce}(Y_0, Y_1) \checkmark$$

> **生活类比**: 像两个会计各算半本账，然后把各自的"小计"**加起来**就是"总计"。
> 与 Column Parallel 的"拼接"不同，Row Parallel 是"求和"。

In [ ]:
# ======== Row Parallel Linear 模拟 ========

# 完整权重: [intermediate_size, hidden_size]
W_down_full = torch.randn(intermediate_size, hidden_size)

# 输入: 来自前一层 Column Parallel 的输出 (已切分)
X_parallel = torch.randn(8, intermediate_size)  # 如果输入已经是分片的

# 完整计算 (参考)
Y_ref = X_parallel @ W_down_full
print(f"完整 GEMM: X[{X_parallel.shape}] @ W[{W_down_full.shape}] → Y[{Y_ref.shape}]")

# ========== TP2 切分 ==========
input_shard = intermediate_size // tp_size
W_gpu0 = W_down_full[0:input_shard, :]           # 上半行
W_gpu1 = W_down_full[input_shard:2*input_shard, :] # 下半行

# 关键: 输入也要切分! (input_is_parallel=True)
X_gpu0 = X_parallel[:, 0:input_shard]     # 前半特征
X_gpu1 = X_parallel[:, input_shard:2*input_shard]  # 后半特征

# 每卡独立 GEMM
Y_gpu0 = X_gpu0 @ W_gpu0  # [8, 4096] — 部分和
Y_gpu1 = X_gpu1 @ W_gpu1  # [8, 4096] — 部分和

print(f"\nTP2 Row Parallel:")
print(f"  GPU 0: X_0[{X_gpu0.shape}] @ W_0[{W_gpu0.shape}] → Y_0[{Y_gpu0.shape}] (部分和)")
print(f"  GPU 1: X_1[{X_gpu1.shape}] @ W_1[{W_gpu1.shape}] → Y_1[{Y_gpu1.shape}] (部分和)")

# all-reduce (sum): 求和
Y_reduced = Y_gpu0 + Y_gpu1  # all-reduce 等价于 sum
print(f"  all-reduce(sum) → Y[{Y_reduced.shape}]")

diff = (Y_reduced - Y_ref).abs().max()
print(f"  验证: max diff = {diff:.2e} ✓")

print(f"""
通信开销:
  all-reduce 传输量 = {Y_gpu0.numel() * 2 / 1024:.1f} KB
  all-reduce 比 all-gather 通信量更大 (ring 算法需要 2 轮)
  但 all-reduce 的结果每卡都有完整的 → 不需要后续 gather
""")

## 10.4 vLLM 中的实现: ColumnParallelLinear & RowParallelLinear

vLLM 把上述数学原理封装在两个类中：

In [ ]:
# vLLM ColumnParallelLinear 的 forward (简化版)
# 源码: vllm/model_executor/layers/linear.py:596

class ColumnParallelLinear_Simplified:
    """
    Y = X @ A,  A 按列切分为 [A_0, A_1, ..., A_{tp-1}]
    每卡存 A_i, 计算 Y_i = X @ A_i
    """
    def __init__(self, input_size, output_size, tp_size, gather_output=False):
        self.tp_size = tp_size
        self.gather_output = gather_output
        self.output_size_per_partition = output_size // tp_size
        # 每卡只存 output_size // tp_size 列
        self.weight = torch.randn(self.output_size_per_partition, input_size)

    def forward(self, input_):
        # GEMM: X @ W.T (PyTorch linear 是 Y = X @ W^T + b)
        output_parallel = F.linear(input_, self.weight)  # cuBLAS GEMM

        if self.gather_output and self.tp_size > 1:
            # 如果需要完整输出: all-gather 拼接所有卡的部分
            output = all_gather(output_parallel)  # 通信!
        else:
            # 不 gather: 每卡保留自己的部分 (后续 RowParallel 会处理)
            output = output_parallel

        return output

# vLLM RowParallelLinear 的 forward (简化版)
# 源码: vllm/model_executor/layers/linear.py:1456

class RowParallelLinear_Simplified:
    """
    Y = X @ A + b,  A 按行切分, X 按最后一维切分
    每卡存 A_i, 输入为 X_i, 计算 Y_i = X_i @ A_i (部分和)
    """
    def __init__(self, input_size, output_size, tp_size, reduce_results=True):
        self.tp_size = tp_size
        self.reduce_results = reduce_results
        self.input_size_per_partition = input_size // tp_size
        # 每卡只存 input_size // tp_size 行
        self.weight = torch.randn(output_size, self.input_size_per_partition)

    def forward(self, input_parallel):
        # input_parallel 已经是分片的 (来自前面的 ColumnParallel)

        # GEMM: X_i @ W_i^T (部分和)
        output_parallel = F.linear(input_parallel, self.weight)  # cuBLAS GEMM

        if self.reduce_results and self.tp_size > 1:
            # all-reduce: 所有卡的部分和求和
            output = all_reduce(output_parallel)  # 通信!
        else:
            output = output_parallel

        return output

# 模拟 all-gather 和 all-reduce
def all_gather(tensor):
    """模拟: 拼接所有卡的 tensor (这里只是 identity)"""
    return tensor

def all_reduce(tensor):
    """模拟: 所有卡求和 (这里只是 identity)"""
    return tensor

print("vLLM 的两种并行线性层:")
print()
print("ColumnParallelLinear:")
print("  权重: 每卡存 output_size // tp_size 列")
print("  GEMM: Y_i = X @ W_i  (完整输入, 部分输出列)")
print("  通信: all-gather (可选, 大多数情况不需要)")
print()
print("RowParallelLinear:")
print("  权重: 每卡存 input_size // tp_size 行")
print("  GEMM: Y_i = X_i @ W_i  (分片输入, 完整输出形状但是部分和)")
print("  通信: all-reduce (必须, 求和得到完整结果)")

## 10.5 Llama 的完整 TP 数据流 — 每层只需 2 次 all-reduce

这是 vLLM 中 Llama 模型的 TP 通信与 GEMM 配合的**完整流程**:

```
                    Llama Decoder Layer (TP2)
                    ========================

输入: hidden_states [tokens, 4096]  ← 每卡都有完整的

 ┌─── Attention Block ──────────────────────────────────────────┐
 │                                                              │
 │  qkv_proj (ColumnParallel, gather_output=False)              │
 │    GPU 0: X @ W_qkv_0 → [tokens, (16+4+4)×128] = [T, 3072] │
 │    GPU 1: X @ W_qkv_1 → [tokens, (16+4+4)×128] = [T, 3072] │
 │    ⚡ 无通信! 每卡只处理自己的 heads                         │
 │                                                              │
 │  Attention (每卡独立处理自己的 heads)                         │
 │    GPU 0: FlashAttention(Q_0, K_0, V_0) → [T, 16×128]       │
 │    GPU 1: FlashAttention(Q_1, K_1, V_1) → [T, 16×128]       │
 │    ⚡ 无通信! GQA 使得 KV head 也可以切分                    │
 │                                                              │
 │  o_proj (RowParallel, reduce_results=True)                   │
 │    GPU 0: attn_out_0 @ W_o_0 → [T, 4096]  (部分和)          │
 │    GPU 1: attn_out_1 @ W_o_1 → [T, 4096]  (部分和)          │
 │    🔴 all-reduce! → 每卡得到完整的 [T, 4096]                │
 │                                                              │
 └──────────────────────────────── 1 次 all-reduce ─────────────┘
                              ↓
               hidden_states + residual (每卡完整)
                              ↓
 ┌─── MLP Block ────────────────────────────────────────────────┐
 │                                                              │
 │  gate_up_proj (MergedColumnParallel, gather_output=False)    │
 │    GPU 0: X @ [W_gate_0; W_up_0] → [T, 2×5504]             │
 │    GPU 1: X @ [W_gate_1; W_up_1] → [T, 2×5504]             │
 │    ⚡ 无通信! 每卡处理一半的 intermediate_size               │
 │                                                              │
 │  SiLU + Mul (每卡独立)                                       │
 │    GPU 0: SiLU(gate_0) * up_0 → [T, 5504]                   │
 │    GPU 1: SiLU(gate_1) * up_1 → [T, 5504]                   │
 │    ⚡ 无通信! 元素级操作在分片上独立                          │
 │                                                              │
 │  down_proj (RowParallel, reduce_results=True)                │
 │    GPU 0: activated_0 @ W_down_0 → [T, 4096]  (部分和)      │
 │    GPU 1: activated_1 @ W_down_1 → [T, 4096]  (部分和)      │
 │    🔴 all-reduce! → 每卡得到完整的 [T, 4096]                │
 │                                                              │
 └──────────────────────────────── 1 次 all-reduce ─────────────┘
                              ↓
               hidden_states + residual (每卡完整)

每个 Transformer 层: 2 次 all-reduce + 4 次 GEMM (每卡)
```

In [ ]:
# ======== 完整模拟: Llama TP2 一层的 GEMM + 通信 ========

tokens = 8
batch = tokens

# 模拟两张 GPU 的权重 (TP2)
# ---- Attention ----
# QKV: ColumnParallel (按输出维度切)
total_heads = 32
heads_per_gpu = total_heads // tp_size  # 16
kv_heads = 8
kv_heads_per_gpu = kv_heads // tp_size  # 4
qkv_out_per_gpu = (heads_per_gpu + 2 * kv_heads_per_gpu) * head_dim  # (16+4+4)*128 = 3072

W_qkv_gpu0 = torch.randn(qkv_out_per_gpu, hidden_size)
W_qkv_gpu1 = torch.randn(qkv_out_per_gpu, hidden_size)

# O: RowParallel (按输入维度切)
W_o_gpu0 = torch.randn(hidden_size, heads_per_gpu * head_dim)
W_o_gpu1 = torch.randn(hidden_size, heads_per_gpu * head_dim)

# ---- MLP ----
inter_per_gpu = intermediate_size // tp_size  # 5504

# Gate+Up: MergedColumnParallel
W_gate_up_gpu0 = torch.randn(2 * inter_per_gpu, hidden_size)
W_gate_up_gpu1 = torch.randn(2 * inter_per_gpu, hidden_size)

# Down: RowParallel
W_down_gpu0 = torch.randn(hidden_size, inter_per_gpu)
W_down_gpu1 = torch.randn(hidden_size, inter_per_gpu)

# ========== 前向传播 ==========
X = torch.randn(batch, hidden_size)
print("=" * 65)
print(f"Llama TP2 一层前向 (tokens={tokens}, hidden={hidden_size})")
print("=" * 65)

# Step 1: QKV Projection (ColumnParallel, 无通信)
qkv_0 = F.linear(X, W_qkv_gpu0)  # GPU 0 GEMM
qkv_1 = F.linear(X, W_qkv_gpu1)  # GPU 1 GEMM
print(f"\n1. QKV Proj (ColumnParallel):")
print(f"   GPU 0: [{batch},{hidden_size}] @ [{hidden_size},{qkv_out_per_gpu}] → [{batch},{qkv_out_per_gpu}]")
print(f"   GPU 1: [{batch},{hidden_size}] @ [{hidden_size},{qkv_out_per_gpu}] → [{batch},{qkv_out_per_gpu}]")
print(f"   通信: 无 ⚡")

# Step 2: Attention (每卡独立)
attn_out_0 = torch.randn(batch, heads_per_gpu * head_dim)  # 模拟
attn_out_1 = torch.randn(batch, heads_per_gpu * head_dim)
print(f"\n2. Attention (每卡独立):")
print(f"   GPU 0: {heads_per_gpu} heads 独立计算")
print(f"   GPU 1: {heads_per_gpu} heads 独立计算")
print(f"   通信: 无 ⚡")

# Step 3: O Projection (RowParallel, all-reduce!)
o_out_0 = F.linear(attn_out_0, W_o_gpu0)  # 部分和
o_out_1 = F.linear(attn_out_1, W_o_gpu1)  # 部分和
o_full = o_out_0 + o_out_1  # 模拟 all-reduce
print(f"\n3. O Proj (RowParallel):")
print(f"   GPU 0: [{batch},{heads_per_gpu*head_dim}] @ [{heads_per_gpu*head_dim},{hidden_size}] → [{batch},{hidden_size}] (部分和)")
print(f"   GPU 1: [{batch},{heads_per_gpu*head_dim}] @ [{heads_per_gpu*head_dim},{hidden_size}] → [{batch},{hidden_size}] (部分和)")
print(f"   🔴 all-reduce(sum) → [{batch},{hidden_size}] (完整)")
print(f"   通信量: {batch * hidden_size * 2 / 1024:.1f} KB")

# Step 4: Gate+Up (MergedColumnParallel, 无通信)
gate_up_0 = F.linear(o_full, W_gate_up_gpu0)
gate_up_1 = F.linear(o_full, W_gate_up_gpu1)
print(f"\n4. Gate+Up Proj (MergedColumnParallel):")
print(f"   GPU 0: [{batch},{hidden_size}] @ [{hidden_size},{2*inter_per_gpu}] → [{batch},{2*inter_per_gpu}]")
print(f"   通信: 无 ⚡")

# Step 5: SiLU+Mul (每卡独立)
gate_0, up_0 = gate_up_0.chunk(2, dim=-1)
activated_0 = torch.nn.functional.silu(gate_0) * up_0
gate_1, up_1 = gate_up_1.chunk(2, dim=-1)
activated_1 = torch.nn.functional.silu(gate_1) * up_1
print(f"\n5. SiLU+Mul (每卡独立):")
print(f"   GPU 0: SiLU(gate_0) * up_0 → [{batch},{inter_per_gpu}]")
print(f"   通信: 无 ⚡")

# Step 6: Down (RowParallel, all-reduce!)
down_0 = F.linear(activated_0, W_down_gpu0)  # 部分和
down_1 = F.linear(activated_1, W_down_gpu1)  # 部分和
down_full = down_0 + down_1  # 模拟 all-reduce
print(f"\n6. Down Proj (RowParallel):")
print(f"   GPU 0: [{batch},{inter_per_gpu}] @ [{inter_per_gpu},{hidden_size}] → [{batch},{hidden_size}] (部分和)")
print(f"   🔴 all-reduce(sum) → [{batch},{hidden_size}] (完整)")
print(f"   通信量: {batch * hidden_size * 2 / 1024:.1f} KB")

print(f"""
\n总结:
  GEMM 次数 (每卡): 4 次 (QKV, O, Gate+Up, Down)
  all-reduce 次数: 2 次 (O proj 后 + Down proj 后)
  all-reduce 总传输量: {2 * batch * hidden_size * 2 / 1024:.1f} KB
""")

## 10.6 为什么 Column + Row 配对可以最小化通信？

```
关键洞察: Column 和 Row 是"天然配对"

ColumnParallel 的输出:
  GPU 0 有结果的"前半列" → [T, out_size/2]
  GPU 1 有结果的"后半列" → [T, out_size/2]

如果下一层是 RowParallel (input_is_parallel=True):
  Row 需要输入按最后一维切分 → 正好就是 Column 的输出格式!
  → 不需要任何通信, 直接衔接!

配对模式:
  Column(无通信) → Row(all-reduce) = 只需 1 次通信
  ↑ 这就是 Megatron-LM 论文的核心发明

如果不配对 (全用 Column + all-gather):
  Column(all-gather) → Column(all-gather) = 每层 2 次 all-gather
  通信量更大, 延迟更高

vLLM 中的配对:
  Attention:  QKV_proj (Column) → 中间不需要通信 → O_proj (Row, all-reduce)
  MLP:        Gate+Up (Column) → 中间不需要通信 → Down (Row, all-reduce)
```

In [ ]:
# 直观理解: Column 输出 "自然衔接" Row 输入

print("Column → Row 配对的数据流:")
print()
print("Step 1: ColumnParallel (gate_up_proj)")
print("  GPU 0 计算 [gate_0; up_0]  shape: [T, 2 × inter//2]")
print("  GPU 1 计算 [gate_1; up_1]  shape: [T, 2 × inter//2]")
print("  → 各卡的输出是 intermediate 维度的不同 partition")
print()
print("Step 2: SiLU + Mul (在分片上独立)")
print("  GPU 0: activated_0 = SiLU(gate_0) * up_0  shape: [T, inter//2]")
print("  GPU 1: activated_1 = SiLU(gate_1) * up_1  shape: [T, inter//2]")
print()
print("Step 3: RowParallel (down_proj, input_is_parallel=True)")
print("  GPU 0: activated_0 @ W_down_0  → [T, hidden]  (部分和)")
print("  GPU 1: activated_1 @ W_down_1  → [T, hidden]  (部分和)")
print("  → all-reduce(sum) 得到完整输出")
print()
print("关键: Step 1 的输出格式 (按 inter 维度切分)")
print("      恰好就是 Step 3 需要的输入格式 (按 input 维度切分)")
print("      → 零通信衔接! 🔥")

## 10.7 MoE 的 TP vs EP 通信

MoE 模型有两种并行策略，通信模式完全不同:

### 策略 1: Tensor Parallel (TP) — 切分每个 expert 的权重

```
TP 切分 MoE (与标准 MLP 类似):
  每个 expert 的权重按列/行切分
  所有 GPU 都有所有 expert 的一部分

  GPU 0: Expert_0 上半权重, Expert_1 上半权重, ..., Expert_7 上半权重
  GPU 1: Expert_0 下半权重, Expert_1 下半权重, ..., Expert_7 下半权重

  通信: all-reduce (与标准 MLP 相同)
  优点: 通信量固定, 与 expert 数无关
  缺点: 每张卡必须存所有 expert (虽然每个只存一半)
```

### 策略 2: Expert Parallel (EP) — 不同卡存不同 expert

```
EP 切分 MoE:
  不同 GPU 持有不同的 expert

  GPU 0: Expert_0, Expert_1, Expert_2, Expert_3 (完整权重)
  GPU 1: Expert_4, Expert_5, Expert_6, Expert_7 (完整权重)

  通信: all-to-all
  ① 根据路由结果, 把 token 发送到持有对应 expert 的 GPU
  ② 各 GPU 计算自己的 expert GEMM
  ③ 把结果发回原来的 GPU

  优点: 每卡存更少的权重 (适合 expert 数很多的模型)
  缺点: all-to-all 通信量取决于路由分布
```

In [ ]:
# MoE 通信对比

num_experts = 8
moe_inter = 2048

print("MoE TP vs EP 通信对比:")
print()
print(f"=== TP (Tensor Parallel, tp_size=2) ===")
print(f"  每卡 expert 权重: {num_experts} × [inter/2, hidden] = {num_experts} × [{moe_inter//2}, {hidden_size}]")
print(f"  每卡 GEMM: 每个 expert 算一半 → 所有 expert 都参与")
print(f"  通信: 1 次 all-reduce")
print(f"  通信量: {batch * hidden_size * 2 / 1024:.1f} KB (固定, 与 expert 数无关)")
print()
print(f"=== EP (Expert Parallel, ep_size=2) ===")
print(f"  每卡 expert 权重: {num_experts//2} × [inter, hidden] = {num_experts//2} × [{moe_inter}, {hidden_size}]")
print(f"  每卡 GEMM: 只算分配到本卡的 expert → 更少 GEMM 次数")
print(f"  通信: all-to-all (发送 token 到对应 expert 的 GPU)")
print(f"  通信量: 取决于路由 (可能很不均匀)")
print()
print("=== 何时选择哪种? ===")
print("  少量 expert (如 Mixtral 8 experts): TP 更好 (通信固定)")
print("  大量 expert (如 DeepSeek-V3 256 experts): EP 必须 (TP 放不下)")
print("  实际中常用 TP + EP 混合: TP 切 expert 内部, EP 切 expert 之间")

## 10.8 通信与计算的重叠 (Overlap)

在高性能推理中，通信与 GEMM 可以**重叠执行**:

```
没有重叠 (序列执行):
  ┌──── GEMM ────┐  ┌── all-reduce ──┐  ┌──── 下一层 GEMM ────┐
  │              │  │                │  │                      │
  0              t1 t1               t2 t2                     t3
  总时间 = GEMM + all-reduce + next_GEMM

有重叠 (通信与计算重叠):
  ┌──── GEMM ────┐
  │              │  ┌──── 下一层 GEMM ────┐
  │   ┌── all-reduce ──┐                  │
  0   t1         t2     t3                 t4
  总时间 ≈ GEMM + max(all-reduce, next_GEMM)  → 通信被隐藏!
```

### vLLM 中的重叠机制

In [ ]:
# vLLM 支持的通信-计算重叠:
# 源码: vllm/distributed/parallel_state.py

print("vLLM 的通信优化策略:")
print()
print("1. NCCL all-reduce 是异步的:")
print("   torch.distributed.all_reduce(tensor, async_op=True)")
print("   → 返回一个 handle, 可以稍后 wait()")
print("   → 在 wait 之前可以做其他计算")
print()
print("2. CUDA Graph 中的通信:")
print("   vLLM 支持将 all-reduce 包含在 CUDA Graph 中")
print("   → 减少 CPU-GPU 同步开销")
print()
print("3. 自定义 all-reduce:")
print("   vLLM 使用自定义的 all-reduce 实现 (pynccl / custom_allreduce)")
print("   → 对于小张量, 自定义实现比 NCCL 更快")
print("   → 对于大张量, 使用 NCCL 的 ring/tree 算法")
print()
print("4. 通信量分析 (每 Transformer 层):")

for tp in [2, 4, 8]:
    comm_bytes = 2 * batch * hidden_size * 2  # 2次 all-reduce, FP16
    # Ring all-reduce: 每 GPU 发送/接收 2*(tp-1)/tp * data_size
    ring_bytes = 2 * (tp - 1) / tp * comm_bytes
    print(f"   TP{tp}: all-reduce 有效传输量 = {ring_bytes / 1024:.1f} KB/层 "
          f"(batch={batch}, hidden={hidden_size})")

## 10.9 Bias 在 TP 中的处理

一个容易被忽视的细节: **bias 不能在所有卡上都加！**

```
RowParallelLinear:
  Y = X_0 @ W_0 + X_1 @ W_1 + b
  如果两卡都加 b: Y = (X_0 @ W_0 + b) + (X_1 @ W_1 + b) = 正确值 + b
  → bias 被加了两次! ❌

vLLM 的解决方案:
  只有 rank 0 的 GPU 在 GEMM 中加 bias, 其他卡不加
  → all-reduce 后, bias 只被加了一次 ✓

源码 (linear.py:1472):
  bias_ = None if (self.tp_rank > 0 or self.skip_bias_add) else self.bias
  output_parallel = self.quant_method.apply(self, input_parallel, bias_)
```

In [ ]:
# 演示 bias 重复加的问题
W_full = torch.randn(4, 8)
b = torch.tensor([1.0, 1.0, 1.0, 1.0])
X = torch.randn(2, 8)

# 正确: 完整 GEMM + 1 次 bias
Y_correct = X @ W_full.T + b
print(f"正确结果 (bias 加 1 次): Y[0] = {Y_correct[0].tolist()[:4]}")

# TP2 错误方式: 两卡都加 bias
W_0 = W_full[:, :4]  # 简化: 按行切
W_1 = W_full[:, 4:]
X_0 = X[:, :4]
X_1 = X[:, 4:]

Y_0_wrong = X_0 @ W_0.T + b  # GPU 0 加了 bias ❌
Y_1_wrong = X_1 @ W_1.T + b  # GPU 1 也加了 bias ❌
Y_wrong = Y_0_wrong + Y_1_wrong  # all-reduce
print(f"错误结果 (bias 加 2 次): Y[0] = {Y_wrong[0].tolist()[:4]}")

# TP2 正确方式: 只有 rank 0 加 bias
Y_0_right = X_0 @ W_0.T + b   # GPU 0 (rank=0) 加 bias ✓
Y_1_right = X_1 @ W_1.T       # GPU 1 (rank=1) 不加 bias ✓
Y_right = Y_0_right + Y_1_right
print(f"正确结果 (rank 0 加 bias): Y[0] = {Y_right[0].tolist()[:4]}")
print(f"验证: max diff = {(Y_correct - Y_right).abs().max():.2e} ✓")

## 10.10 完整通信图谱

```
                vLLM TP 通信与 GEMM 完整图谱
                ============================

层类型                  GEMM 实现              通信操作         通信量
─────────────────────────────────────────────────────────────────────
QKV Proj               cuBLAS (Column)         无              0
K Proj                 cuBLAS (Column)         无              0
V Proj                 cuBLAS (Column)         无              0
Attention Q*K, attn*V  FlashAttention          无              0
O Proj                 cuBLAS (Row)            all-reduce      2·T·H bytes
Gate Proj              cuBLAS (Column)         无              0
Up Proj                cuBLAS (Column)         无              0
SiLU + Mul             —                       无              0
Down Proj              cuBLAS (Row)            all-reduce      2·T·H bytes
─────────────────────────────────────────────────────────────────────
每层总计: 2 次 all-reduce, 总传输 4·T·H bytes (FP16)

MoE 变体:
  TP MoE:              Triton fused_moe        all-reduce      2·T·H bytes
  EP MoE:              Triton fused_moe        all-to-all      变化
─────────────────────────────────────────────────────────────────────

T = num_tokens, H = hidden_size

对于 Llama-70B (H=8192):
  每层通信: 4 × T × 8192 = 32768·T bytes
  80 层总通信: 2,621,440·T bytes ≈ 2.5 MB × T
  Batch=1 (decode): ~2.5 MB/step
  Batch=128 (prefill): ~320 MB/step
```

## 10.11 源码映射表

| 本章内容 | vLLM 源码位置 | 说明 |
|----------|--------------|------|
| ColumnParallelLinear | `vllm/model_executor/layers/linear.py:429` | 按列切分, 可选 all-gather |
| RowParallelLinear | `vllm/model_executor/layers/linear.py:1308` | 按行切分, all-reduce |
| QKVParallelLinear | `vllm/model_executor/layers/linear.py:923` | QKV 合并的 ColumnParallel |
| MergedColumnParallel | `vllm/model_executor/layers/linear.py:626` | Gate+Up 合并版 |
| TP all-reduce | `vllm/distributed/communication_op.py:12` | `tensor_model_parallel_all_reduce` |
| TP all-gather | `vllm/distributed/communication_op.py:17` | `tensor_model_parallel_all_gather` |
| Llama Attention | `vllm/model_executor/models/llama.py:123` | QKV(Column) → Attn → O(Row) |
| Llama MLP | `vllm/model_executor/models/llama.py:80` | Gate+Up(Column) → Down(Row) |
| FusedMoE Layer | `vllm/model_executor/layers/fused_moe/layer.py:275` | MoE 的 TP/EP |
| Bias 处理 | `vllm/model_executor/layers/linear.py:1472` | 只有 rank 0 加 bias |